# Retail Shelf Monitoring — Exploratory Analysis

This notebook provides tools for:
1. Visualizing shelf images and detections
2. Inspecting the FAISS index and embeddings
3. Analyzing alerts from a run
4. Profiling inference speed


In [ ]:
# Setup — add project root to path
import sys
from pathlib import Path

# Adjust this to your project root if needed
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is run from notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import cv2
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

print('Imports OK')

## 1. Visualize Shelf Image with Detections

In [ ]:
# Load and display a shelf image with annotated detections
from shelf_monitor.frameworks.model_registry import ModelRegistry, ModelMode

models_dir = PROJECT_ROOT / 'models'
reg = ModelRegistry(models_dir=models_dir)
print(f'Model mode: {reg.mode}')
print(f'Detector:   {reg.detector_path()}')
print(f'Embedding:  {reg.embedding_path()}')
print(f'FAISS:      {reg.index_path()}')

In [ ]:
# Generate a synthetic shelf image to explore (no real models needed)
from scripts.make_demo_data import make_all_demo_data
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    ref_img_path, video_path, crops_dir, planogram = make_all_demo_data(
        out_dir=Path(tmpdir) / 'demo'
    )
    ref_img = cv2.imread(str(ref_img_path))
    ref_img_rgb = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.imshow(ref_img_rgb)
plt.title('Synthetic Reference Shelf Image')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Inspect the FAISS Index

In [ ]:
# Only works after training and loading models
index_path = PROJECT_ROOT / 'models' / 'sku_index.faiss'
labels_path = PROJECT_ROOT / 'models' / 'sku_labels.json'

if index_path.exists() and labels_path.exists():
    import faiss
    index = faiss.read_index(str(index_path))
    with open(labels_path) as f:
        labels = json.load(f)
    
    print(f'FAISS index ntotal: {index.ntotal}')
    print(f'FAISS index dimension: {index.d}')
    print(f'Number of SKU classes: {len(labels)}')
    print(f'First 5 labels: {dict(list(labels.items())[:5])}')
else:
    print('No FAISS index found. Train models first and copy to models/')

## 3. Analyze Alerts from a Demo Run

In [ ]:
# Run the demo and capture outputs
import tempfile
from scripts.run_demo import run_full_demo

with tempfile.TemporaryDirectory() as tmpdir:
    out_dir = Path(tmpdir) / 'demo_run'
    ret = run_full_demo(out_dir=out_dir)
    
    with open(out_dir / 'alerts.json') as f:
        alerts = json.load(f)
    with open(out_dir / 'metrics_report.json') as f:
        metrics = json.load(f)

print(f'Return code: {ret}')
print(f'Total alerts: {len(alerts)}')
print(f'Metrics: {json.dumps(metrics, indent=2)}')

In [ ]:
# Visualize alert breakdown
from collections import Counter

alert_type_counts = Counter(a['alert_type'] for a in alerts)

fig, ax = plt.subplots(figsize=(8, 4))
types = list(alert_type_counts.keys())
counts = list(alert_type_counts.values())
bars = ax.bar(types, counts, color=['#e74c3c', '#f39c12', '#3498db'])
ax.bar_label(bars, padding=3)
ax.set_title('Alert Type Distribution')
ax.set_xlabel('Alert Type')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Model Registry Status

In [ ]:
from shelf_monitor.frameworks.model_registry import ModelRegistry, ModelMode

reg = ModelRegistry(models_dir=PROJECT_ROOT / 'models')

mode_descriptions = {
    ModelMode.BASELINE: '⚠️  BASELINE — Using stock yolo11n (no custom training)',
    ModelMode.POSITION_ONLY: '🟡 POSITION-ONLY — Fine-tuned detector, no SKU recognition',
    ModelMode.FULL: '✅ FULL — Complete pipeline with SKU identification',
}

print(mode_descriptions.get(reg.mode, 'Unknown mode'))
print()
print('Contract files:')
contract_files = [
    'product_detector.pt', 'product_detector.onnx',
    'sku_embedding.pt', 'sku_embedding.onnx',
    'sku_index.faiss', 'sku_labels.json', 'model_manifest.json'
]
for fname in contract_files:
    path = PROJECT_ROOT / 'models' / fname
    status = '✓' if path.exists() else '✗'
    print(f'  {status} {fname}')

## 5. Inference Speed Profiling

In [ ]:
import time

# Create a dummy 640x480 BGR frame for benchmarking
dummy_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

print('Profiling YOLO forward pass on dummy frame...')
try:
    from shelf_monitor.adaptors.ml.yolo_detector import YOLODetector
    from shelf_monitor.frameworks.config import MLConfig
    
    cfg = MLConfig()
    detector = YOLODetector(config=cfg, models_dir=PROJECT_ROOT / 'models')
    
    # Warmup
    for _ in range(3):
        detector.detect(dummy_frame)
    
    # Benchmark
    N = 20
    t0 = time.perf_counter()
    for _ in range(N):
        detector.detect(dummy_frame)
    elapsed = time.perf_counter() - t0
    
    ms_per_frame = elapsed / N * 1000
    fps = N / elapsed
    print(f'  Avg per frame: {ms_per_frame:.1f} ms')
    print(f'  Effective FPS: {fps:.1f}')
except Exception as e:
    print(f'Profiling skipped: {e}')

## 6. Config Inspection

In [ ]:
from shelf_monitor.frameworks.config import AppConfig

config = AppConfig.load(PROJECT_ROOT / 'config.yaml')
print('Active configuration:')
print(json.dumps(config.model_dump(), indent=2, default=str))